In [1]:
# === GRAZIOSO SALVARE RESCUE DASHBOARD ===
# === Paulo Bezerra ===

# --- 1) Imports ---
import pandas as pd
from jupyter_dash import JupyterDash
from dash import html, dcc, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
from animal_shelter import AnimalShelter
from bson import ObjectId

# --- 2) Connect to MongoDB using CRUD module ---
shelter = AnimalShelter("aacuser", "AACpass123")

# --- 3) Load Data and Convert ObjectIds ---
data = list(shelter.read({}))
for item in data:
    if '_id' in item and isinstance(item['_id'], ObjectId):
        item['_id'] = str(item['_id'])
df = pd.DataFrame(data)

# --- 4) Initialize Dash App ---
app = JupyterDash(__name__)

# --- 5) Define Layout ---
app.layout = html.Div([
    html.Img(src='/assets/grazioso_logo.png', style={'height': '100px'}),
    html.H1("GRAZIOSO SALVARE RESCUE DASHBOARD"),
    html.H3("Paulo Bezerra"),

    # Filter widget: radio buttons for rescue type
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water Rescue'},
            {'label': 'Mountain/Wilderness Rescue', 'value': 'Mountain/Wilderness Rescue'},
            {'label': 'Disaster/Individual Tracking', 'value': 'Disaster/Individual Tracking'},
            {'label': 'Reset (Show All)', 'value': 'Reset'}
        ],
        value='Reset',
        labelStyle={'display': 'inline-block', 'margin-right': '20px'}
    ),

    # Data table
    dash_table.DataTable(
        id='datatable',
        columns=[{"name": i, "id": i} for i in df.columns],
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'}
    ),

    html.H2("Rescue Locations"),
    dl.Map(
        id="map",
        center=[30.75, -97.75],
        zoom=10,
        style={'width': '1000px', 'height': '500px'}
    )
])

# --- 6) Callback: filter data & update table + map ---
@app.callback(
    [Output('datatable', 'data'),
     Output('map', 'children')],
    [Input('filter-type', 'value')]
)
def update_dashboard(rescue_type):
    """
    Filters data based on selected rescue type
    and generates map markers for matching animals.
    """
    if rescue_type == 'Water Rescue':
        dff = df[df['breed'].str.contains('Labrador Retriever', case=False, na=False)]
    elif rescue_type == 'Mountain/Wilderness Rescue':
        dff = df[df['breed'].str.contains('German Shepherd', case=False, na=False)]
    elif rescue_type == 'Disaster/Individual Tracking':
        dff = df[df['breed'].str.contains('Bloodhound', case=False, na=False)]
    else:
        dff = df

    markers = [
        dl.Marker(
            position=[row['location_lat'], row['location_long']],
            children=dl.Tooltip(f"{row['breed']} ({row['animal_id']})")
        )
        for _, row in dff.iterrows()
        if not pd.isna(row['location_lat']) and not pd.isna(row['location_long'])
    ]

    return dff.to_dict('records'), markers

# --- 7) Run App ---
app.run_server(mode='inline')
